# 🕵️‍♂️ Phase 0: Pre-Cleaning Data Analysis

This notebook performs a quick "health check" on the raw CSV datasets before the ETL pipeline.

**Refactored for modularity, readability, and type safety.**

**Goals:**
1.  Detect missing values and data quality issues.
2.  Infer correct data types (int vs float).
3.  Identify columns to drop.
4.  Identify almost-empty rows (irrelevant data).

In [19]:
import pandas as pd
import glob
import os
from IPython.display import display, Markdown
from typing import Optional, List

# Configuration
DATA_DIR: str = "../../data/raw"
SAMPLE_SIZE: int = 100000

## 🛠️ Atomic Analysis Functions

### 1. Load Data
Safely reads a sample of the CSV file, skipping bad lines.

In [20]:
def load_sample_data(file_path: str, sample_size: int = SAMPLE_SIZE) -> Optional[pd.DataFrame]:
    """Reads a sample of the CSV file safely, reporting bad lines with details."""

    # Mutable state for the closure
    stats = {"bad_lines": 0, "logged": 0}

    def bad_line_handler(line: List[str]) -> None:
        stats["bad_lines"] += 1

        # Log only the first 5 errors to avoid flooding output
        if stats["logged"] < 5:
            print(f"⚠️ Bad line #{stats['bad_lines']} detected (Fields: {len(line)}): {line[:5]}...")
            stats["logged"] += 1
        elif stats["logged"] == 5:
            print(f"⚠️ ... more bad lines detected (suppressing output) ...")
            stats["logged"] += 1
        return None

    try:
        # engine='python' is required because 'pyarrow' does not support 'nrows' (sampling)
        # and the default 'c' engine does not support callable 'on_bad_lines'.
        df = pd.read_csv(file_path, nrows=sample_size, on_bad_lines=bad_line_handler, engine='python')
        if stats["bad_lines"] > 0:
            print(f"❌ Total Bad Lines Skipped: {stats['bad_lines']}")
        return df
    except Exception as e:
        print(f"🔴 ERROR reading file: {e}")
        return None

### 2. Check Shape
Reports the dimensions (rows and columns) of the loaded sample.

In [21]:
def check_shape(df: pd.DataFrame) -> None:
    """Reports dataset dimensions."""

    print(f"📏 Shape (Sample): {df.shape[0]} rows, {df.shape[1]} columns")

### 3. Check Missing Values
Identifies columns with null values and visualizes the percentage of missing data.

In [22]:
def check_missing_values(df: pd.DataFrame) -> None:
    """Identifies and reports columns with missing values."""

    missing = df.isnull().sum()
    missing_percent = (missing / len(df)) * 100

    missing_data = pd.DataFrame({
        'Missing Count': missing,
        'Percentage': missing_percent
    })

    missing_data = missing_data[missing_data['Missing Count'] > 0].sort_values(by='Percentage', ascending=False)

    if not missing_data.empty:
        print("\n⚠️ Missing Values Detected:")
        display(missing_data)
    else:
        print("\n✅ No Missing Values Found (in sample)")

### 4. Check Data Types
Lists the inferred data types for each column to help identify casting needs.

In [23]:
def check_data_types(df: pd.DataFrame) -> None:
    """Reports inferred data types."""

    print("\nℹ️ Inferred Data Types:")
    print(df.dtypes)

### 5. Check Duplicates (Dynamic)
Attempts to infer the Primary Key and checks for uniqueness.

In [24]:
def check_duplicates(df: pd.DataFrame) -> None:
    """Dynamically finds the likely primary key and checks for duplicates."""

    # Priority list for potential Primary Keys
    possible_pks: List[str] = ['mal_id', 'id', 'username', 'user_id', 'character_mal_id', 'person_mal_id']

    target_pk: Optional[str] = None

    # 1. Exact match from priority list
    for pk in possible_pks:
        if pk in df.columns:
            target_pk = pk
            break

    # 2. Heuristic: First column ending with '_id' if no match yet
    if not target_pk:
        for col in df.columns:
            if col.lower().endswith('_id') or col.lower().endswith('id'):
                target_pk = str(col)
                break

    if target_pk:
        duplicates = df.duplicated(target_pk).sum()
        if duplicates > 0:
            print(f"\n❌ Duplicate Primary Key candidate '{target_pk}': {duplicates} rows")
        else:
            print(f"\n✅ Primary Key candidate '{target_pk}' is unique")
    else:
        print("\n⚠️ Check skipped: Could not automatically infer a Primary Key.")

### 6. Check Almost Empty Rows
Detects 'ghost rows' that have >80% missing values and prints their few populated columns.

In [25]:
def check_almost_empty_rows(df: pd.DataFrame, threshold: float = 0.8) -> None:
    """Detects rows that are mostly empty (>threshold missing)."""

    empty_mask = df.isnull().mean(axis=1) > threshold
    empty_rows = df[empty_mask]

    if not empty_rows.empty:
        print(f"\n🗑️ Found {len(empty_rows)} rows with >{threshold*100:.0f}% missing values")
        print("   First 5 instances of 'Ghost Rows':")
        
        for idx, row in empty_rows.head(5).iterrows():
            populated = row[row.notnull()].index.tolist()
            # Truncate populated list if needed checking
            print(f"   - Row {idx}: Only has data in {populated}")
    else:
        print("\n✅ No 'almost-empty' rows found")

## 🎬 Orchestrator

Runs the atomic checks sequentially for a given file.

In [26]:
def analyze_file(file_path: str) -> None:
    """Analyzes a dataset for common issues."""

    filename = os.path.basename(file_path)
    display(Markdown(f"### 📄 Analysis: `{filename}`"))
    print(f"{'='*40}")

    df = load_sample_data(file_path)
    if df is not None:
        check_shape(df)
        check_duplicates(df)
        check_missing_values(df)
        check_almost_empty_rows(df, threshold=0.8)
        check_data_types(df)
        print(f"{'='*40}\n")

## 🚀 Run Analysis

In [27]:
csv_files: List[str] = glob.glob(os.path.join(DATA_DIR, "*.csv"))
print(f"Found {len(csv_files)} CSV files in {DATA_DIR}\n")

for csv_file in csv_files:
    analyze_file(csv_file)

Found 13 CSV files in ../data/raw



### 📄 Analysis: `characters.csv`

📏 Shape (Sample): 100000 rows, 7 columns

❌ Duplicate Primary Key candidate 'character_mal_id': 286 rows

⚠️ Missing Values Detected:


,Missing Count,Percentage
about,58959,58.959
name_kanji,33759,33.759



✅ No 'almost-empty' rows found

ℹ️ Inferred Data Types:
character_mal_id     int64
url                 object
name                object
name_kanji          object
image               object
favorites            int64
about               object
dtype: object



...

📏 Shape (Sample): 100000 rows, 2 columns

❌ Duplicate Primary Key candidate 'mal_id': 91487 rows

✅ No Missing Values Found (in sample)

✅ No 'almost-empty' rows found

ℹ️ Inferred Data Types:
mal_id                   int64
recommendation_mal_id    int64
dtype: object



### 📄 Analysis: `stats.csv`

📏 Shape (Sample): 28955 rows, 27 columns

✅ Primary Key candidate 'mal_id' is unique

⚠️ Missing Values Detected:


,Missing Count,Percentage
score_1_votes,430,1.485063
score_1_percentage,430,1.485063
score_2_votes,430,1.485063
score_2_percentage,430,1.485063
score_3_votes,430,1.485063
score_3_percentage,430,1.485063
score_4_votes,430,1.485063
score_4_percentage,430,1.485063
score_5_votes,430,1.485063
score_5_percentage,430,1.485063



✅ No 'almost-empty' rows found

ℹ️ Inferred Data Types:
mal_id                   int64
watching                 int64
completed                int64
on_hold                  int64
dropped                  int64
plan_to_watch            int64
total                    int64
score_1_votes          float64
score_1_percentage     float64
score_2_votes          float64
score_2_percentage     float64
score_3_votes          float64
score_3_percentage     float64
score_4_votes          float64
score_4_percentage     float64
score_5_votes          float64
score_5_percentage     float64
score_6_votes          float64
score_6_percentage     float64
score_7_votes          float64
score_7_percentage     float64
score_8_votes          float64
score_8_percentage     float64
score_9_votes          float64
score_9_percentage     float64
score_10_votes         float64
score_10_percentage    float64
dtype: object

